# Synthesis generation audit

See when 10-generation experiments first found their best candidate and whether generations 11–20 helped longer experiments.

The notebook reads saved results from SQLite in read-only mode. Set `DATABASE_PATH` to use a separate study database.

In [25]:
import os
import sqlite3
import sys
from functools import partial
from pathlib import Path

import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, select
from sqlalchemy.orm import sessionmaker

cwd = Path.cwd()
root = next(path for path in (cwd, *cwd.parents) if (path / "src/evolution").is_dir())
for import_path in (root, root / "src"):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from evolution.application.synthesis.generation_analysis import (
    GenerationAnalysis,
    analyze_generations,
)
from evolution.infra.storage.synthesis.repository import SQLiteSynthesisRepository
from notebooks.analysis.plotting.style import FONT_FAMILY, get_solver_color
from shared.infra.database.tables import ExperimentORM

DATABASE_PATH = Path(os.environ.get("AAD_SYNTHESIS_DB", root / "data/db.sqlite3"))
FIGURE_STYLE = dict(template="plotly_white", font_family=FONT_FAMILY)

MODEL_IDS = None  # Optional tuple of exact recorded names, e.g. ("model-id",).

In [26]:
database_path = (root / DATABASE_PATH).resolve()
uri = f"{database_path.as_uri()}?mode=ro"
engine = create_engine("sqlite://", creator=partial(sqlite3.connect, uri, uri=True))
try:
    ids_query = select(ExperimentORM.id).order_by(ExperimentORM.id)
    if MODEL_IDS is not None:
        ids_query = ids_query.where(ExperimentORM.llm_name.in_(MODEL_IDS))
    with engine.connect() as connection:
        experiment_ids = list(connection.execute(ids_query).scalars())

    repository = SQLiteSynthesisRepository(sessionmaker(engine))
    analysis = analyze_generations(repository.load_by_ids(experiment_ids))
finally:
    engine.dispose()

print(f"Database: {database_path}")
print(f"Recorded experiments selected: {len(experiment_ids)}")

Database: /Users/nicolaibrahim/Desktop/proj/AAD_LLM/data/db.sqlite3
Recorded experiments selected: 2079


## When did 10-generation experiments find their best candidate?

An **experiment** is one LLaMEA run for a fixed problem and configuration. A **generation** is one candidate attempt in that run. Each bar shows the percentage of included experiments whose lowest recorded error first occurred at that generation. Ties count at the earliest generation.

In [27]:
def show_generation_chart(analysis: GenerationAnalysis) -> None:
    ten = analysis.distribution[
        (analysis.distribution["generations"] == 10) & analysis.distribution["model"].isna()
    ]
    ten_runs = analysis.runs[analysis.runs["generations"] == 10]
    ten_exclusions = analysis.exclusions[analysis.exclusions["generations"] == 10]
    valid = int(ten_runs["best_generation"].count())
    no_score = len(ten_runs) - valid
    incomplete = int(ten_exclusions["reason"].eq("not_completed").sum())
    wrong_count = int(ten_exclusions["reason"].eq("iteration_count_mismatch").sum())
    coverage_note = (
        f"Included experiments: {valid:,}. Excluded from the 10-generation chart: "
        f"{incomplete:,} unfinished or failed, "
        f"{wrong_count:,} completed with missing or extra generation records, "
        f"and {no_score:,} completed with no valid score."
    )

    if not valid:
        print("No eligible experiments with a valid score to plot.")
        print(coverage_note)
        return

    generation_figure = go.Figure(
        go.Bar(
            x=ten["generation"],
            y=ten["best_fraction"],
            customdata=ten["best_count"],
            text=ten["best_fraction"],
            texttemplate="%{text:.1%}",
            textposition="outside",
            marker_color=["#315d89"] * 9 + ["#b94e21"],
            hovertemplate=(
                "Generation %{x}<br>%{customdata:,} of "
                f"{valid:,} experiments (%{{y:.1%}})<extra></extra>"
            ),
        )
    )
    generation_figure.update_xaxes(title="LLaMEA generation", dtick=1, range=[0.5, 10.5])
    generation_figure.update_yaxes(
        title="% of experiments",
        tickformat=".0%",
        range=[0, max(0.05, ten["best_fraction"].max() * 1.20)],
    )
    generation_figure.update_layout(
        title="Generation where the best candidate first appeared",
        height=410,
        **FIGURE_STYLE,
    )
    display(generation_figure)
    print(coverage_note)


show_generation_chart(analysis)


Included experiments: 1,446. Excluded from the 10-generation chart: 553 unfinished or failed, 80 completed with missing or extra generation records, and 0 completed with no valid score.


## Did generations 11–20 improve the result?

Each dot represents one completed 20-generation experiment with a valid score at both cutoffs. On the left, dots below the diagonal improved. On the right, a larger positive difference means a larger drop in error.


In [28]:
def show_paired_comparison(analysis: GenerationAnalysis) -> None:
    pairs = analysis.pairs.dropna(subset=["error_at_10", "error_at_20"])

    if pairs.empty:
        print("No completed 20-generation experiments have valid scores at both cutoffs.")
        return

    paired_figure = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("Best error at each cutoff", "Absolute improvement"),
    )
    for model, group in pairs.groupby("model", sort=True):
        trace_style = dict(
            mode="markers",
            marker_color=get_solver_color(model),
            name=model,
            legendgroup=model,
        )
        paired_figure.add_scatter(
            x=group["error_at_10"],
            y=group["error_at_20"],
            text=group["experiment_id"],
            hovertemplate="Experiment %{text}<br>10: %{x:.4g}<br>20: %{y:.4g}<extra></extra>",
            row=1,
            col=1,
            **trace_style,
        )
        paired_figure.add_scatter(
            x=group["experiment_id"],
            y=group["absolute_improvement"],
            text=group["experiment_id"],
            hovertemplate="Experiment %{text}<br>Improvement: %{y:.4g}<extra></extra>",
            showlegend=False,
            row=1,
            col=2,
            **trace_style,
        )
    error_limit = pairs["error_at_10"].max()
    paired_figure.add_shape(
        type="line", x0=0, y0=0, x1=error_limit, y1=error_limit,
        line=dict(color="#64748B", dash="dot"), row=1, col=1,
    )
    paired_figure.update_xaxes(title_text="Best error at generation 10", row=1, col=1)
    paired_figure.update_yaxes(title_text="Best error at generation 20", row=1, col=1)
    paired_figure.update_xaxes(title_text="Experiment ID", row=1, col=2)
    paired_figure.update_yaxes(title_text="Error at 10 minus error at 20", row=1, col=2)
    pooled = analysis.paired_summary.loc[analysis.paired_summary["model"].isna()].iloc[0]
    print(
        f"{pooled['improved']:,} of {pooled['comparable']:,} comparable experiments improved "
        f"({pooled['improved_fraction']:.1%}). Median error reduction across those "
        f"comparable experiments: {pooled['median_absolute_improvement']:.3g}."
    )
    paired_figure.update_layout(
        title="Error reduction from generations 11–20",
        height=490,
        **FIGURE_STYLE,
    )
    display(paired_figure)


show_paired_comparison(analysis)


No completed 20-generation experiments have valid scores at both cutoffs.
